In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, SequentialSampler, Subset, Dataset
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import os
from tqdm.auto import tqdm
import warnings
import copy
import math
import time
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
#dataset/////////////////////////////////
class MyDataset(Dataset):
    def __init__(self, factor, label):
        self.factor = factor
        self.label = label
        # merge factor and label on 'ts_code' and 'trade_date'
        self.factor['trade_date'] = pd.to_datetime(self.factor['trade_date'], format='%Y%m')
        self.label['trade_date'] = pd.to_datetime(self.label['trade_date'], format='%Y%m')
        self.label['trade_date'] = self.label['trade_date'] - pd.DateOffset(months=1)
        self.data_origin = pd.merge(self.factor, self.label, on=['ts_code', 'trade_date'], how='left')
        self.data_origin = self.data_origin.sort_values(['trade_date','ts_code'], ascending=[True, True])
        self.data_origin.dropna(inplace=True)
        self.data_origin.drop(columns = ['ts_code', 'trade_date'], inplace=True)
        self.data_origin = self.data_origin.applymap(lambda x: float(x))
        # winsorize the data
        self.data_origin = self.winsorize_data(self.data_origin, lower=0.1, upper=0.9)
        # convert to tensor
        if torch.cuda.is_available():
            self.data_origin = torch.tensor(self.data_origin.values, dtype=torch.float32, device='cuda:0')
        else:
            self.data_origin = torch.tensor(self.data_origin.values, dtype=torch.float32, device='cpu')
        return

    def __len__(self):
        return len(self.data_origin)

    # factor, highest_return, lowest_return, close_return
    def __getitem__(self, idx):
        self.factor = self.data_origin[idx, :-3]
        self.highest_return = self.data_origin[idx, -3]
        self.lowest_return = self.data_origin[idx, -2]
        self.close_return = self.data_origin[idx, -1]
        return self.factor, self.highest_return, self.lowest_return, self.close_return
    
    # def get_factors_norm_params(self, train_size):
    #     train_indice = int(len(self.data_origin) * train_size)
    #     train_data = self.data_origin[:train_indice, :-3]
    #     self.factor_min = train_data.min(dim=0, keepdim=True)[0]
    #     self.factor_max = train_data.max(dim=0, keepdim=True)[0]
    #     self.factor_mean = train_data.mean(dim=0, keepdim=True)
    #     self.factor_std = train_data.std(dim=0, keepdim=True)
    #     return
        
    def get_labels_norm_params(self, train_size):
        train_indice = int(len(self.data_origin) * train_size)
        train_data = self.data_origin[:train_indice, -3:]
        # self.label_min = train_data.min(dim=0, keepdim=True)[0]
        # self.label_max = train_data.max(dim=0, keepdim=True)[0]
        self.label_mean = train_data.mean(dim=0, keepdim=True)
        self.label_std = train_data.std(dim=0, keepdim=True)
        return
    
    def winsorize_data(self, df, lower=0.04, upper=0.96):
        for col in df.select_dtypes(include=['float64', 'int64']).columns:
            lower_bound = df[col].quantile(lower)
            upper_bound = df[col].quantile(upper)
            df[col] = df[col].clip(lower_bound, upper_bound)
        return df



class MyDataLoader(DataLoader):
    def __init__(self, dataset, batch_size=32, shuffle=True, num_workers=0, train_size=0.5, test_size=0.1):
        self.dataset = dataset
        if train_size + test_size > 1.0:
            raise ValueError("train_size + test_size must be less than 1.0")
        self.train_size = train_size
        self.test_size = test_size
        # super(MyDataLoader, self).__init__(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers)
        self.shuffle = shuffle
        self.batch_size = batch_size
        self.num_workers = num_workers

    def get_norm_params(self, train_size=None):
        if train_size is not None:
            self.train_size = train_size
        # get normalization parameters
        # self.dataset.get_factors_norm_params(train_size=self.train_size)
        self.dataset.get_labels_norm_params(train_size=self.train_size)
        # self.factor_min = self.dataset.factor_min
        # self.factor_max = self.dataset.factor_max
        # self.factor_mean = self.dataset.factor_mean
        # self.factor_std = self.dataset.factor_std
        # self.label_min = self.dataset.label_min
        # self.label_max = self.dataset.label_max
        self.label_mean = self.dataset.label_mean
        self.label_std = self.dataset.label_std
        return


    def get_train_loader(self, train_size=None):
        if train_size is not None:
            self.train_size = train_size
        # get normalization parameters
        self.get_norm_params(train_size=self.train_size)
        # create train subset
        train_indice = int(len(self.dataset) * self.train_size)
        indices = list(range(len(self.dataset)))
        train_indices = indices[:train_indice]
        dataset_copy = copy.deepcopy(self.dataset)
        train_subset = Subset(dataset_copy, train_indices)
        # normalize the factors
        # train_subset_factors_norm = train_subset.dataset.data_origin[:train_indice, :-3]
        # train_subset_factors_norm = (train_subset_factors_norm - self.factor_mean) / (self.factor_std + 1e-8)
        # train_subset.dataset.data_origin[:train_indice, :-3] = train_subset_factors_norm
        # normalize the targets
        train_subset_labels_norm = train_subset.dataset.data_origin[:train_indice, -3:]
        train_subset_labels_norm = (train_subset_labels_norm - self.label_mean) / (self.label_std + 1e-8)
        train_subset.dataset.data_origin[:train_indice, -3:] = train_subset_labels_norm
        return DataLoader(train_subset, batch_size=self.batch_size, shuffle=self.shuffle, num_workers=self.num_workers)
        

    def get_test_loader(self, test_size=None):
        if test_size is not None:
            self.test_size = test_size       
        train_indice = int(len(self.dataset) * self.train_size)
        test_indice = train_indice + int(len(self.dataset) * self.test_size)
        indices = list(range(len(self.dataset)))
        test_indices = indices[train_indice:test_indice]
        dataset_copy = copy.deepcopy(self.dataset)
        test_subset = Subset(dataset_copy, test_indices)
        # normalize the factors
        # test_subset_factors_norm = test_subset.dataset.data_origin[train_indice:test_indice, :-3]
        # test_subset_factors_norm = (test_subset_factors_norm - self.factor_mean) / (self.factor_std + 1e-8)
        # test_subset.dataset.data_origin[train_indice:test_indice, :-3] = test_subset_factors_norm
        # normalize the targets
        test_subset_labels_norm = test_subset.dataset.data_origin[train_indice:test_indice, -3:]
        test_subset_labels_norm = (test_subset_labels_norm - self.label_mean) / (self.label_std + 1e-8)
        test_subset.dataset.data_origin[train_indice:test_indice, -3:] = test_subset_labels_norm
        return DataLoader(test_subset, batch_size=self.batch_size, shuffle=self.shuffle, num_workers=self.num_workers)





class LoadData():
    def __init__(self, factor, label, batch_size=32, shuffle=False, num_workers=0, train_size=0.5, test_size=0.1):
        self.dataset = MyDataset(factor, label)
        self.dataloader = MyDataLoader(self.dataset, 
                                       batch_size=batch_size, 
                                       shuffle=shuffle, 
                                       num_workers=num_workers, 
                                       train_size=train_size, 
                                       test_size=test_size)
    

    def get_train_loader(self, train_size=None):
        if train_size is not None:
            self.dataloader.train_size = train_size
        _ = self.dataloader.get_train_loader(train_size=self.dataloader.train_size)
        self.label_mean = self.dataloader.label_mean
        self.label_std = self.dataloader.label_std
        return _

    def get_test_loader(self, test_size=None):
        if test_size is not None:
            self.dataloader.test_size = test_size
        return self.dataloader.get_test_loader(test_size=self.dataloader.test_size)



class RiskMetrics:
    def __init__(self, risk_free_rate=0.01, data_frequency='monthly'):
        self.risk_free_rate = risk_free_rate
        
        if data_frequency == 'daily':
            self.periods_per_year = 252
        elif data_frequency == 'weekly':
            self.periods_per_year = 52
        elif data_frequency == 'monthly':
            self.periods_per_year = 12
        elif data_frequency == 'quarterly':
            self.periods_per_year = 4
        else:
            raise ValueError("Supported frequencies: 'daily', 'weekly', 'monthly', 'quarterly'")
    
    def calculate_metrics(self, returns):
        """计算年化风险指标"""
        # 剔除return为0的值
        returns = np.array(returns)
        returns = returns[returns != 0]

        # 基础统计
        mean_return = np.mean(returns)
        std_return = np.std(returns)
        
        # 年化指标
        annualized_return = mean_return * self.periods_per_year
        annualized_volatility = std_return * np.sqrt(self.periods_per_year)
        
        # 夏普比率
        if annualized_volatility > 0:
            sharpe_ratio = (annualized_return - self.risk_free_rate) / annualized_volatility
        else:
            sharpe_ratio = 0
        
        # 最大回撤
        cumulative_returns = np.cumprod(1 + returns/100) - 1 
        peak = np.maximum.accumulate(cumulative_returns)
        drawdown = (peak - cumulative_returns) / (peak + 1e-8)
        max_drawdown = -np.max(drawdown)
        
        # 其他指标
        win_rate = np.sum(returns > 0) / len(returns)
        
        return {
            'mean_return': mean_return,
            'std_return': std_return,
            'annualized_return': annualized_return,
            'annualized_volatility': annualized_volatility,
            'sharpe_ratio': sharpe_ratio,
            'max_drawdown': max_drawdown,
            'win_rate': win_rate,
            'data_frequency': f'{self.periods_per_year} periods/year'
        }


class ClippedLeakyReLU(nn.Module):
    def __init__(self, min_val=-0.2, max_val=0.2, negative_slope=0.3):
        super().__init__()
        self.min_val = min_val
        self.max_val = max_val
        self.negative_slope = negative_slope
        
    def forward(self, x):
        x = torch.where(x >= 0, x, self.negative_slope * x)
        x = torch.clamp(x, self.min_val, self.max_val)
        return x

class ScaledTanh(nn.Module):
    def __init__(self, scale=0.2):
        super().__init__()
        self.scale = scale
    
    def forward(self, x):
        return torch.tanh(x) * self.scale


class PositionalEncoding(nn.Module):
    """位置编码模块"""
    
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        
        self.register_buffer('pe', pe)

    def forward(self, x):
        """
        参数:
            x: Tensor, shape [batch_size, seq_len, d_model]
        """
        x = x + self.pe[:x.size(1), :].transpose(0, 1)
        return self.dropout(x)





In [ ]:
# model///////////////////////////////////////////////////////////////////
class LinearRegression(nn.Module):
    def __init__(self, input_dim, output_dim=1, target =3, model_name = "linear_regression"):
        super(LinearRegression, self).__init__()
        
        self.layers = nn.Sequential(
                nn.BatchNorm1d(input_dim),
                nn.Linear(input_dim, output_dim)
            )
        
        self._initialize_weights()
        self.optimizer = optim.Adam(self.parameters(), lr=0.01)
        self.target = target
        self.model_name = model_name
        self.loss_fn = nn.MSELoss()
        if not os.path.exists("model/checkpoints"):
            os.makedirs("model/checkpoints")
        if not os.path.exists("model/final_models"):
            os.makedirs("model/final_models")
        self.checkpoint_path = f"model/checkpoints/{self.model_name}_checkpoint.pth"
        self.model_path = f"model/final_models/{self.model_name}.pth"
        if torch.cuda.is_available():
            self.device = torch.device("cuda")
        else:
            self.device = torch.device("cpu")
        self.to(self.device)

    def forward(self, x):
        return self.layers(x)
    
    def reset_model(self):
        self._initialize_weights()
        self.optimizer = optim.Adam(self.parameters(), lr=0.01)
        return

    def _initialize_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0, std=1)
                nn.init.constant_(module.bias, 0)
            elif isinstance(module, nn.BatchNorm1d):
                nn.init.constant_(module.weight, 1)
                nn.init.constant_(module.bias, 0)

    
    def train_step(self, train_loader,criterion=None):
        criterion = self.loss_fn
        self.train()
        total_loss = 0.0
        for train_data in train_loader:
            inputs = train_data[0].to(self.device)
            targets = train_data[self.target].to(self.device)
            if targets.dim() == 1:
                targets = targets.unsqueeze(1)
            self.optimizer.zero_grad()
            outputs = self.forward(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item()
        return total_loss / len(train_loader)


    def fit(self, train_loader, epochs=100, resume_training=False, patience=5, criterion=None):
        start_epoch = 0
        no_improvement_count = 0
        if resume_training:
            start_epoch = self._load_checkpoint()
            print(f"Resuming training from epoch {start_epoch + 1}")

        prev_loss = float('inf')
        for epoch in tqdm(range(start_epoch, epochs), colour='#FA6780'):
            train_loss = self.train_step(train_loader)
            # print(f"Epoch [{epoch + 1}/{epochs}], Loss: {train_loss:.4f}")

            # Save checkpoint every 10 epochs
            if (epoch + 1) % 10 == 0:
                self._save_checkpoint(epoch)
                print(f"Epoch [{epoch + 1}/{epochs}], Loss: {train_loss:.4f}")

            # if loss is not improving for 5 epochs, stop training
            if round(train_loss,4) < round(prev_loss,4) - 0.001:
                prev_loss = train_loss
                no_improvement_count = 0  
                self.save_model()  
            else:
                no_improvement_count += 1
                if no_improvement_count >= patience:
                    # print("Early stopping triggered due to no improvement in loss.")
                    break
        

    # Predict method & reverse normalization
    def predict(self, test_loader, label_mean, label_std):
        self.eval()
        pred_list = []
        act_list = []
        with torch.no_grad():
            for test_data in test_loader:
                data = test_data[0].to(self.device)
                act_high = test_data[1].to(self.device)
                act_low = test_data[2].to(self.device)
                act_close = test_data[3].to(self.device)
                if act_high.dim() == 1:
                    act_high = act_high.unsqueeze(1)
                if act_low.dim() == 1:
                    act_low = act_low.unsqueeze(1)
                if act_close.dim() == 1:
                    act_close = act_close.unsqueeze(1)
                act = torch.cat([act_high, act_low, act_close], dim=1)
                act = act * label_std + label_mean
                act_list.append(act.cpu().numpy())
                if data.dim() == 1:
                    data = data.unsqueeze(1)
                pred = self(data)
                # reverse normalization
                pred = pred * label_std + label_mean
                pred_list.append(pred.cpu().numpy())
        pred = np.concatenate(pred_list, axis=0)
        act = np.concatenate(act_list, axis=0)
        return pred, act


    # nondemeaned R^2 evaluation
    def evaluate(self, test_loader):
        self.eval()
        total_sse = 0.0
        total_ss = 0.0
        with torch.no_grad():
            for test_data in test_loader:
                inputs = test_data[0].to(self.device)
                targets = test_data[self.target].to(self.device)
                if targets.dim() == 1:
                    targets = targets.unsqueeze(1)
                outputs = self(inputs)
                # targets >=0
                targets = torch.clamp(targets, min=0)
                outputs = torch.clamp(outputs, min=0)
                # Calculate sum of squared errors and total sum of squares
                sse = torch.sum((targets - outputs) ** 2)
                ss = torch.sum(targets ** 2)
                total_sse += sse.item()
                total_ss += ss.item()
        if total_ss < 1e-8:
            return 0
        else:
            return 1 - total_sse / total_ss


    def _save_checkpoint(self, epoch):
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'loss': self.loss_fn
        }
        torch.save(checkpoint, self.checkpoint_path)

    def _load_checkpoint(self):
        if os.path.exists(self.checkpoint_path):
            checkpoint = torch.load(self.checkpoint_path, weights_only=False, map_location=self.device)
            self.load_state_dict(checkpoint['model_state_dict'])
            self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            return checkpoint['epoch']
        else:
            print("No checkpoint found. Starting from scratch.")
            return 0

    def save_model(self, path=None):
        if path is None:
            path = self.model_path
        os.makedirs(os.path.dirname(path), exist_ok=True) 
        torch.save(self.state_dict(), path)

    def load_model(self, path=None):
        if path is None:
            path = self.model_path
        self.load_state_dict(torch.load(path, map_location=self.device))


class ElasticNet(nn.Module):
    def __init__(self, input_dim, output_dim=1, target=3, alpha=1.0, l1_ratio=0.5, model_name = "ElasticNet"):
        super(ElasticNet, self).__init__()
        self.layers = nn.Sequential(
                nn.BatchNorm1d(input_dim),
                nn.Linear(input_dim, output_dim)
            )
        
        self._initialize_weights()
        self.optimizer = optim.Adam(self.parameters(), lr=0.01)
        self.model_name = model_name
        self.alpha = alpha
        self.l1_ratio = l1_ratio
        self.target = target
        self.loss_fn = nn.MSELoss()
        if not os.path.exists("model/checkpoints"):
            os.makedirs("model/checkpoints")
        if not os.path.exists("model/final_models"):
            os.makedirs("model/final_models")
        self.checkpoint_path = f"model/checkpoints/{self.model_name}_checkpoint.pth"
        self.model_path = f"model/final_models/{self.model_name}.pth"
        if torch.cuda.is_available():
            self.device = torch.device("cuda")
        else:
            self.device = torch.device("cpu")
        self.to(self.device)

    def forward(self, x):
        return self.layers(x)

    def elastic_net_loss(self, outputs, targets):
        mse_loss = self.loss_fn(outputs, targets)
        l1_reg = 0
        l2_reg = 0
        
        # 修复：遍历所有模块而不是引用不存在的 self.linear
        for module in self.modules():
            if isinstance(module, nn.Linear):
                l1_reg += torch.sum(torch.abs(module.weight))
                l2_reg += torch.sum(module.weight ** 2)
        
        elastic_reg = self.alpha * (self.l1_ratio * l1_reg + (1 - self.l1_ratio) * l2_reg)
        return mse_loss + elastic_reg
    

    def _initialize_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0, std=1)
                nn.init.constant_(module.bias, 0)
            elif isinstance(module, nn.BatchNorm1d):
                nn.init.constant_(module.weight, 1)
                nn.init.constant_(module.bias, 0)

    def reset_model(self):
        # 修复：重新初始化所有权重而不是引用不存在的 self.linear
        self._initialize_weights()
        self.optimizer = optim.Adam(self.parameters(), lr=0.01)
        return
    

    def train_step(self, train_loader,criterion=None):
        criterion = self.elastic_net_loss
        self.train()
        total_loss = 0.0
        for train_data in train_loader:
            inputs = train_data[0].to(self.device)
            targets = train_data[self.target].to(self.device)
            if targets.dim() == 1:
                targets = targets.unsqueeze(1)
            self.optimizer.zero_grad()
            outputs = self.forward(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item()
        return total_loss / len(train_loader)

    def fit(self, train_loader, epochs=100, resume_training=False, patience=5, criterion='elastic_net'):
        start_epoch = 0
        no_improvement_count = 0
        if resume_training:
            start_epoch = self._load_checkpoint()
            print(f"Resuming training from epoch {start_epoch + 1}")

        prev_loss = float('inf')
        for epoch in tqdm(range(start_epoch, epochs), colour='#FA6780'):
            train_loss = self.train_step(train_loader, criterion=criterion)

            # Save checkpoint every 10 epochs
            if (epoch + 1) % 10 == 0:
                self._save_checkpoint(epoch)
                print(f"Epoch [{epoch + 1}/{epochs}], Loss: {train_loss:.4f}")

            # if loss is not improving for 5 epochs, stop training
            if round(train_loss,4) < round(prev_loss,4) - 0.001:
                prev_loss = train_loss
                no_improvement_count = 0   
                self.save_model() 
            else:
                no_improvement_count += 1
                if no_improvement_count >= patience:
                    # print("Early stopping triggered due to no improvement in loss.")
                    break
        

    # Predict method & reverse normalization
    def predict(self, test_loader, label_mean, label_std):
        self.eval()
        pred_list = []
        act_list = []
        with torch.no_grad():
            for test_data in test_loader:
                data = test_data[0].to(self.device)
                act_high = test_data[1].to(self.device)
                act_low = test_data[2].to(self.device)
                act_close = test_data[3].to(self.device)
                if act_high.dim() == 1:
                    act_high = act_high.unsqueeze(1)
                if act_low.dim() == 1:
                    act_low = act_low.unsqueeze(1)
                if act_close.dim() == 1:
                    act_close = act_close.unsqueeze(1)
                act = torch.cat([act_high, act_low, act_close], dim=1)
                act = act * label_std + label_mean
                act_list.append(act.cpu().numpy())
                if data.dim() == 1:
                    data = data.unsqueeze(1)
                pred = self(data)
                # reverse normalization
                pred = pred * label_std + label_mean
                pred_list.append(pred.cpu().numpy())
        pred = np.concatenate(pred_list, axis=0)
        act = np.concatenate(act_list, axis=0)
        return pred, act
        
    # nondemeaned R^2 evaluation
    def evaluate(self, test_loader):
        self.eval()
        total_sse = 0.0
        total_ss = 0.0
        with torch.no_grad():
            for test_data in test_loader:
                inputs = test_data[0].to(self.device)
                targets = test_data[self.target].to(self.device)
                if targets.dim() == 1:
                    targets = targets.unsqueeze(1)
                outputs = self(inputs)
                # targets >=0
                targets = torch.clamp(targets, min=0)
                outputs = torch.clamp(outputs, min=0)
                # Calculate sum of squared errors and total sum of squares
                sse = torch.sum((targets - outputs) ** 2)
                ss = torch.sum(targets ** 2)
                total_sse += sse.item()
                total_ss += ss.item()
        if total_ss < 1e-8:
            return 0
        else:
            return 1 - total_sse / total_ss

    def _save_checkpoint(self, epoch):
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'loss': self.elastic_net_loss
        }
        torch.save(checkpoint, self.checkpoint_path)

    def _load_checkpoint(self):
        if os.path.exists(self.checkpoint_path):
            checkpoint = torch.load(self.checkpoint_path, weights_only=False, map_location=self.device)
            self.load_state_dict(checkpoint['model_state_dict'])
            self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            return checkpoint['epoch']
        else:
            print("No checkpoint found. Starting from scratch.")
            return 0

    def save_model(self, path=None):
        if path is None:
            path = self.model_path
        os.makedirs(os.path.dirname(path), exist_ok=True) 
        torch.save(self.state_dict(), path)
        # print (f"Model successfully saved to {path}")

    def load_model(self, path=None):
        if path is None:
            path = self.model_path
        self.load_state_dict(torch.load(path, map_location=self.device))
        # print (f"Model successfully loaded from {path}")
    

class NN(nn.Module):
    def __init__(self, input_dim, output_dim=1, target=3, alpha=1.0, l1_ratio=0.5, layer = 2, model_name = "NN"):
        super(NN, self).__init__()
        
        if layer == 1:
            self.layers = nn.Sequential(
                nn.BatchNorm1d(input_dim),
                nn.Linear(input_dim, output_dim)
            )
        elif layer == 2:
            self.layers = nn.Sequential(
                nn.BatchNorm1d(input_dim),
                nn.Linear(input_dim, 32),
                nn.BatchNorm1d(32),
                nn.LeakyReLU(0.3),
                nn.Linear(32, output_dim)
            )
        elif layer == 3:
            self.layers = nn.Sequential(
                nn.BatchNorm1d(input_dim),
                nn.Linear(input_dim, 32),
                nn.BatchNorm1d(32),
                nn.LeakyReLU(0.3),
                nn.Linear(32, 16),
                nn.BatchNorm1d(16),
                nn.LeakyReLU(0.3),
                nn.Linear(16, output_dim)
            )
        elif layer == 4:
            self.layers = nn.Sequential(
                nn.BatchNorm1d(input_dim),
                nn.Linear(input_dim, 32),
                nn.BatchNorm1d(32),
                nn.LeakyReLU(0.3),
                nn.Linear(32, 16),
                nn.BatchNorm1d(16),
                nn.LeakyReLU(0.3),
                nn.Linear(16, 8),
                nn.BatchNorm1d(8),
                nn.LeakyReLU(0.3),
                nn.Linear(8, output_dim)
            )
        elif layer == 5:
            self.layers = nn.Sequential(
                nn.BatchNorm1d(input_dim),
                nn.Linear(input_dim, 32),
                nn.BatchNorm1d(32),
                nn.LeakyReLU(0.3),
                nn.Linear(32, 16),
                nn.BatchNorm1d(16),
                nn.LeakyReLU(0.3),
                nn.Linear(16, 8),
                nn.BatchNorm1d(8),
                nn.LeakyReLU(0.3),
                nn.Linear(8, 4),
                nn.BatchNorm1d(4),
                nn.LeakyReLU(0.3),
                nn.Linear(4, output_dim)
            )
        else:
            raise ValueError("Unsupported number of layers. Supported values are 1 to 5.")
        
        self._initialize_weights()
        self.optimizer = optim.Adam(self.parameters(), lr=0.01)
        self.loss_fn = nn.MSELoss()
        self.model_name = model_name + f"_{layer}Layers"
        self.alpha = alpha
        self.l1_ratio = l1_ratio
        self.target = target
        if not os.path.exists("model/checkpoints"):
            os.makedirs("model/checkpoints")
        if not os.path.exists("model/final_models"):
            os.makedirs("model/final_models")
        self.checkpoint_path = f"model/checkpoints/{self.model_name}_checkpoint.pth"
        self.model_path = f"model/final_models/{self.model_name}.pth"

        if torch.cuda.is_available():
            self.device = torch.device("cuda")
        else:
            self.device = torch.device("cpu")
        self.to(self.device)

    def _initialize_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0, std=1)
                nn.init.constant_(module.bias, 0)
            elif isinstance(module, nn.BatchNorm1d):
                nn.init.constant_(module.weight, 1)
                nn.init.constant_(module.bias, 0)

    def forward(self, x):
        return self.layers(x)

    def elastic_net_loss(self, outputs, targets):
        mse_loss = self.loss_fn(outputs, targets)
        l1_reg = 0
        l2_reg = 0
        for module in self.modules():
            if isinstance(module, nn.Linear):
                l1_reg += torch.sum(torch.abs(module.weight))
                l2_reg += torch.sum(module.weight ** 2)
        
        elastic_reg = self.alpha * (self.l1_ratio * l1_reg + (1 - self.l1_ratio) * l2_reg)
        return mse_loss + elastic_reg


    def reset_model(self):
        self._initialize_weights()
        self.optimizer = optim.Adam(self.parameters(), lr=0.01)
        return


    def train_step(self, train_loader, criterion=None):
        if criterion is None:
            criterion = self.loss_fn
        elif criterion == "elastic_net":
            criterion = self.elastic_net_loss
        else:
            raise ValueError("Unsupported criterion. Supported values are 'elastic_net', 'mse_rs', or None (default MSE).")

        self.train()
        total_loss = 0.0

        for train_data in train_loader:
            inputs = train_data[0].to(self.device)
            targets = train_data[self.target].to(self.device)
            if targets.dim() == 1:
                targets = targets.unsqueeze(1)
            self.optimizer.zero_grad()
            outputs = self.forward(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item()
        return total_loss / len(train_loader)


    def fit(self, train_loader, epochs=100, resume_training=False, patience=5, criterion=None):
        start_epoch = 0
        no_improvement_count = 0
        if resume_training:
            start_epoch = self._load_checkpoint()
            print(f"Resuming training from epoch {start_epoch + 1}")

        prev_loss = float('inf')
        for epoch in tqdm(range(start_epoch, epochs), colour='#FA6780'):
            train_loss = self.train_step(train_loader, criterion)
            # print(f"Epoch [{epoch + 1}/{epochs}], Loss: {train_loss:.4f}")
            # Save checkpoint every 10 epochs
            if (epoch + 1) % 10 == 0:
                self._save_checkpoint(epoch)
                print(f"Epoch [{epoch + 1}/{epochs}], Loss: {train_loss:.4f}")

            # if loss is not improving for 5 epochs, stop training
            if round(train_loss,4) < round(prev_loss,4) - 0.001:
                prev_loss = train_loss
                no_improvement_count = 0  
                # 保存最佳模型
                self.save_model()  
            else:
                no_improvement_count += 1
                if no_improvement_count >= patience:
                    # print("Early stopping triggered due to no improvement in loss.")
                    break
        
    # Predict method & reverse normalization
    def predict(self, test_loader, label_mean, label_std):
        self.eval()
        pred_list = []
        act_list = []
        with torch.no_grad():
            for test_data in test_loader:
                data = test_data[0].to(self.device)
                act_high = test_data[1].to(self.device)
                act_low = test_data[2].to(self.device)
                act_close = test_data[3].to(self.device)
                if act_high.dim() == 1:
                    act_high = act_high.unsqueeze(1)
                if act_low.dim() == 1:
                    act_low = act_low.unsqueeze(1)
                if act_close.dim() == 1:
                    act_close = act_close.unsqueeze(1)
                act = torch.cat([act_high, act_low, act_close], dim=1)
                act = act * label_std + label_mean
                act_list.append(act.cpu().numpy())
                if data.dim() == 1:
                    data = data.unsqueeze(1)
                pred = self(data)
                # reverse normalization
                pred = pred * label_std + label_mean
                pred_list.append(pred.cpu().numpy())
        pred = np.concatenate(pred_list, axis=0)
        act = np.concatenate(act_list, axis=0)
        return pred, act
        
    # nondemeaned R^2 evaluation
    def evaluate(self, test_loader):
        self.eval()
        total_sse = 0.0
        total_ss = 0.0
        with torch.no_grad():
            for test_data in test_loader:
                inputs = test_data[0].to(self.device)
                targets = test_data[self.target].to(self.device)
                if targets.dim() == 1:
                    targets = targets.unsqueeze(1)
                outputs = self(inputs)

                # targets >=0
                targets = torch.clamp(targets, min=0)
                outputs = torch.clamp(outputs, min=0)
                
                # Calculate sum of squared errors and total sum of squares
                sse = torch.sum((targets - outputs) ** 2)
                ss = torch.sum(targets ** 2)
                total_sse += sse.item()
                total_ss += ss.item()
        if total_ss < 1e-8:
            return 0
        else:
            return 1 - total_sse / total_ss


    def _save_checkpoint(self, epoch):
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'loss': self.loss_fn
        }
        torch.save(checkpoint, self.checkpoint_path)
        # print(f"Checkpoint saved at epoch {epoch + 1}")

    def _load_checkpoint(self):
        if os.path.exists(self.checkpoint_path):
            checkpoint = torch.load(self.checkpoint_path, weights_only=False, map_location=self.device)
            self.load_state_dict(checkpoint['model_state_dict'])
            self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            return checkpoint['epoch']
        else:
            print("No checkpoint found. Starting from scratch.")
            return 0

    def save_model(self, path=None):
        if path is None:
            path = self.model_path
        os.makedirs(os.path.dirname(path), exist_ok=True) 
        torch.save(self.state_dict(), path)
        # print (f"Model successfully saved to {path}")

    def load_model(self, path=None):
        if path is None:
            path = self.model_path
        self.load_state_dict(torch.load(path, map_location=self.device))
        # print (f"Model successfully loaded from {path}")


class RandomForest:
    def __init__(self, target=3, n_estimators=100, max_depth=None, min_samples_split=5, 
                 min_samples_leaf=2, random_state=42, model_name="RandomForest"):
        from sklearn.ensemble import RandomForestRegressor
        
        self.model = RandomForestRegressor(
            n_estimators=n_estimators,      # 树的数量
            max_depth=max_depth,            # 树的最大深度
            min_samples_split=min_samples_split,  # 分割节点最小样本数
            min_samples_leaf=min_samples_leaf,    # 叶节点最小样本数
            random_state=random_state,      # 随机种子
            n_jobs=-1                      # 使用所有CPU核心
        )
        
        self.model_name = model_name
        self.feature_names = None
        self.target = target
        
        # 创建模型保存目录
        if not os.path.exists("model/checkpoints"):
            os.makedirs("model/checkpoints")
        if not os.path.exists("model/final_models"):
            os.makedirs("model/final_models")
            
        self.checkpoint_path = f"model/checkpoints/{self.model_name}_checkpoint.pkl"
        self.model_path = f"model/final_models/{self.model_name}.pkl"

    def fit(self, train_loader, epochs=None, patience=None, criterion=None):
        """
        训练随机森林模型
        """
        X_train, y_train = self._extract_data_from_loader(train_loader)
        self.model.fit(X_train, y_train)
        self.save_model()
        # print(f"Random Forest trained with {len(X_train)} samples")

    def predict(self, test_loader, label_mean=None, label_std=None):
        """预测并返回与PyTorch模型相同的格式"""
        X_test, y_test = self._extract_data_from_loader(test_loader)
        pred = self.model.predict(X_test)

        # 转换为与PyTorch模型相同的格式
        if pred.ndim == 1:
            pred = pred.reshape(-1, 1)
        if y_test.ndim == 1:
            y_test = y_test.reshape(-1, 1)
        
        # 修复：将 label_mean 和 label_std 转换为 NumPy 数组
        if label_mean is not None and label_std is not None:
            # 确保 label_mean 和 label_std 是 NumPy 数组
            if hasattr(label_mean, 'cpu'):  # 如果是 PyTorch 张量
                label_mean = label_mean.cpu().numpy()
            if hasattr(label_std, 'cpu'):   # 如果是 PyTorch 张量
                label_std = label_std.cpu().numpy()
            
            # 确保维度匹配
            if label_mean.ndim == 0:
                label_mean = label_mean.item()
            if label_std.ndim == 0:
                label_std = label_std.item()
            
            pred = pred * label_std + label_mean
            y_test = y_test * label_std + label_mean
            
        return pred, y_test

    def evaluate(self, test_loader):
        """计算 nondemeaned R²分数"""
        X_test, y_test = self._extract_data_from_loader(test_loader)
        pred = self.model.predict(X_test)
        
        # 确保预测值和目标值非负（与其他模型保持一致）
        y_test = np.clip(y_test, a_min=0, a_max=None)
        pred = np.clip(pred, a_min=0, a_max=None)
        
        # 计算 nondemeaned R²
        # R² = 1 - SSE/SS (其中 SS = Σ(yi)², 不减去均值)
        sse = np.sum((y_test - pred) ** 2)  # 残差平方和
        ss = np.sum(y_test ** 2)            # 总平方和（不去均值）
        
        if ss < 1e-8:
            return 0
        else:
            return 1 - sse / ss

    def _extract_data_from_loader(self, data_loader):
        """从PyTorch DataLoader中提取数据"""
        X_list = []
        y_list = []
        
        for batch in data_loader:
            X_batch = batch[0].numpy()  # 特征
            y_batch = batch[self.target].numpy()  # 目标
            
            X_list.append(X_batch)
            y_list.append(y_batch)
        
        X = np.concatenate(X_list, axis=0)
        y = np.concatenate(y_list, axis=0)
        
        # 确保y是一维数组（sklearn期望的格式）
        if y.ndim > 1:
            y = y.ravel()
            
        return X, y

    def reset_model(self):
        """重置模型（重新初始化）"""
        from sklearn.ensemble import RandomForestRegressor
        
        # 保存原始参数
        params = self.model.get_params()
        self.model = RandomForestRegressor(**params)


    def save_model(self, path=None):
        """保存模型"""
        import joblib
        if path is None:
            path = self.model_path
        
        os.makedirs(os.path.dirname(path), exist_ok=True)
        joblib.dump(self.model, path)
        # print(f"Model successfully saved to {path}")

    def load_model(self, path=None):
        """加载模型"""
        import joblib
        if path is None:
            path = self.model_path
            
        if os.path.exists(path):
            self.model = joblib.load(path)
            # print(f"Model successfully loaded from {path}")
        else:
            print(f"Model file {path} not found")

    def get_feature_importance(self):
        """获取特征重要性"""
        if hasattr(self.model, 'feature_importances_'):
            return self.model.feature_importances_
        else:
            print("Model not trained yet")
            return None

class XGBoost:
    def __init__(self, target=3, n_estimators=100, max_depth=10, learning_rate=0.01, 
                 subsample=0.8, colsample_bytree=0.8, random_state=42, 
                 model_name="XGBoost"):
        """
        XGBoost回归模型
        
        参数:
            target: 目标列索引 (默认3)
            n_estimators: 树的数量 (默认100)
            max_depth: 树的最大深度 (默认3)
            learning_rate: 学习率 (默认0.1)
            subsample: 样本采样比例 (默认0.8)
            colsample_bytree: 特征采样比例 (默认0.8)
            random_state: 随机种子 (默认42)
            model_name: 模型名称 (默认"XGBoost")
        """
        self.model = XGBRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            random_state=random_state,
            n_jobs=-1  # 使用所有CPU核心
        )
        
        self.model_name = model_name
        self.feature_names = None
        self.target = target
        
        # 创建模型保存目录
        os.makedirs("model/checkpoints", exist_ok=True)
        os.makedirs("model/final_models", exist_ok=True)
            
        self.checkpoint_path = f"model/checkpoints/{self.model_name}_checkpoint.pkl"
        self.model_path = f"model/final_models/{self.model_name}.pkl"

    def fit(self, train_loader, epochs=None, patience=None, criterion=None):
        """
        训练XGBoost模型
        
        参数:
            train_loader: 数据加载器 (PyTorch DataLoader格式)
            epochs: 为了接口兼容保留 (XGBoost不需要)
            patience: 为了接口兼容保留
            criterion: 为了接口兼容保留
        """
        X_train, y_train = self._extract_data_from_loader(train_loader)
        self.model.fit(X_train, y_train)
        self.save_model()

    def predict(self, test_loader, label_mean=None, label_std=None):
        """
        预测并返回与PyTorch模型相同的格式
        
        参数:
            test_loader: 测试数据加载器
            label_mean: 标准化均值 (可选)
            label_std: 标准化标准差 (可选)
            
        返回:
            (pred, y_test) 元组，与PyTorch模型输出格式一致
        """
        X_test, y_test = self._extract_data_from_loader(test_loader)
        pred = self.model.predict(X_test)

        # 转换为与PyTorch模型相同的格式
        if pred.ndim == 1:
            pred = pred.reshape(-1, 1)
        if y_test.ndim == 1:
            y_test = y_test.reshape(-1, 1)
        
        # 反标准化处理
        if label_mean is not None and label_std is not None:
            if hasattr(label_mean, 'cpu'):  # 如果是PyTorch张量
                label_mean = label_mean.cpu().numpy()
            if hasattr(label_std, 'cpu'):
                label_std = label_std.cpu().numpy()
            
            pred = pred * label_std + label_mean
            y_test = y_test * label_std + label_mean
            
        return pred, y_test

    def evaluate(self, test_loader):
        """
        计算nondemeaned R²分数
        
        参数:
            test_loader: 测试数据加载器
            
        返回:
            R²分数
        """
        X_test, y_test = self._extract_data_from_loader(test_loader)
        pred = self.model.predict(X_test)
        
        # 确保非负
        y_test = np.clip(y_test, a_min=0, a_max=None)
        pred = np.clip(pred, a_min=0, a_max=None)
        
        # 计算nondemeaned R²
        sse = np.sum((y_test - pred) ** 2)
        ss = np.sum(y_test ** 2)
        
        return 0 if ss < 1e-8 else 1 - sse / ss

    def _extract_data_from_loader(self, data_loader):
        """
        从PyTorch DataLoader中提取数据
        
        参数:
            data_loader: PyTorch DataLoader
            
        返回:
            (X, y) 元组
        """
        X_list, y_list = [], []
        
        for batch in data_loader:
            X_list.append(batch[0].numpy())
            y_list.append(batch[self.target].numpy())
        
        X = np.concatenate(X_list, axis=0)
        y = np.concatenate(y_list, axis=0).ravel()  # XGBoost需要一维目标
        
        return X, y

    def reset_model(self):
        """重置模型（重新初始化）"""
        params = self.model.get_params()
        self.model = XGBRegressor(**params)

    def save_model(self, path=None):
        """
        保存模型到文件
        
        参数:
            path: 自定义保存路径 (可选)
        """
        path = path or self.model_path
        os.makedirs(os.path.dirname(path), exist_ok=True)
        joblib.dump(self.model, path)

    def load_model(self, path=None):
        """
        从文件加载模型
        
        参数:
            path: 自定义加载路径 (可选)
        """
        path = path or self.model_path
        if os.path.exists(path):
            self.model = joblib.load(path)
        else:
            print(f"Model file {path} not found")

    def get_feature_importance(self, importance_type='weight'):
        """
        获取特征重要性
        
        参数:
            importance_type: 
                'weight' - 被使用的次数
                'gain' - 平均信息增益 (默认)
                'cover' - 覆盖的样本数
                
        返回:
            特征重要性数组
        """
        if hasattr(self.model, 'get_score'):
            return self.model.get_booster().get_score(importance_type=importance_type)
        else:
            print("Model not trained yet")
            return None

    def set_params(self, **params):
        """设置模型参数"""
        self.model.set_params(**params)



class K_Means_NN(nn.Module):
    def __init__(self, input_dim, output_dim=1, target=3, n_clusters=10, layer=2, alpha=1.0, l1_ratio=0.5, model_name="K_Means_NN"):
        super(K_Means_NN, self).__init__()
        
        from sklearn.cluster import KMeans
        
        self.input_dim = input_dim
        self.output_dim = output_dim  # 添加这行
        self.n_clusters = n_clusters
        self.layer = layer  # 添加这行
        self.model_name = model_name + f"_{layer}Layers_{n_clusters}Clusters"
        
        # K-means聚类器
        self.kmeans = KMeans(
            n_clusters=self.n_clusters,
            init='k-means++',
            n_init='auto',
            max_iter=300,
            tol=1e-4,
            random_state=42,
            algorithm='lloyd',
            copy_x=True
        )
        self.cluster_labels = None
        self.is_fitted = False
        self.alpha = alpha
        self.l1_ratio = l1_ratio
        self.target = target
        
        # 为每个聚类创建独立的神经网络
        self.cluster_models = nn.ModuleDict()
        for i in range(n_clusters):
            self.cluster_models[f'cluster_{i}'] = self._create_network(input_dim, output_dim, layer)
        
        # 初始化所有聚类模型的权重
        self._initialize_weights()
        
        # 优化器和损失函数
        self.optimizer = optim.Adam(self.parameters(), lr=0.01)
        self.loss_fn = nn.MSELoss()
        
        # 设备设置
        if torch.cuda.is_available():
            self.device = torch.device("cuda")
        else:
            self.device = torch.device("cpu")
        self.to(self.device)
        
        # 模型保存路径
        if not os.path.exists("model/checkpoints"):
            os.makedirs("model/checkpoints")
        if not os.path.exists("model/final_models"):
            os.makedirs("model/final_models")
        self.checkpoint_path = f"model/checkpoints/{self.model_name}_checkpoint.pth"
        self.model_path = f"model/final_models/{self.model_name}.pth"

    def _create_network(self, input_dim, output_dim, layer):
        """为每个聚类创建独立的神经网络 - 修复：返回创建的网络"""
        if layer == 1:
            network = nn.Sequential(
                nn.BatchNorm1d(input_dim),
                nn.Linear(input_dim, output_dim)
            )
        elif layer == 2:
            network = nn.Sequential(
                nn.BatchNorm1d(input_dim),
                nn.Linear(input_dim, 32),
                nn.BatchNorm1d(32),
                nn.LeakyReLU(0.3),
                nn.Linear(32, output_dim)
            )
        elif layer == 3:
            network = nn.Sequential(
                nn.BatchNorm1d(input_dim),
                nn.Linear(input_dim, 32),
                nn.BatchNorm1d(32),
                nn.LeakyReLU(0.3),
                nn.Linear(32, 16),
                nn.BatchNorm1d(16),
                nn.LeakyReLU(0.3),
                nn.Linear(16, output_dim)
            )
        elif layer == 4:
            network = nn.Sequential(
                nn.BatchNorm1d(input_dim),
                nn.Linear(input_dim, 32),
                nn.BatchNorm1d(32),
                nn.LeakyReLU(0.3),
                nn.Linear(32, 16),
                nn.BatchNorm1d(16),
                nn.LeakyReLU(0.3),
                nn.Linear(16, 8),
                nn.BatchNorm1d(8),
                nn.LeakyReLU(0.3),
                nn.Linear(8, output_dim)
            )
        elif layer == 5:
            network = nn.Sequential(
                nn.BatchNorm1d(input_dim),
                nn.Linear(input_dim, 32),
                nn.BatchNorm1d(32),
                nn.LeakyReLU(0.3),
                nn.Linear(32, 16),
                nn.BatchNorm1d(16),
                nn.LeakyReLU(0.3),
                nn.Linear(16, 8),
                nn.BatchNorm1d(8),
                nn.LeakyReLU(0.3),
                nn.Linear(8, 4),
                nn.BatchNorm1d(4),
                nn.LeakyReLU(0.3),
                nn.Linear(4, output_dim)
            )
        else:
            raise ValueError("Unsupported number of layers. Supported values are 1 to 5.")
        
        return network  # 修复：返回创建的网络
    
    def _initialize_weights(self):
        """初始化所有聚类模型的权重"""
        for _, model in self.cluster_models.items():
            for module in model.modules():
                if isinstance(module, nn.Linear):
                    nn.init.normal_(module.weight, mean=0, std=0.1)  # 减小初始化标准差
                    nn.init.constant_(module.bias, 0)
                elif isinstance(module, nn.BatchNorm1d):
                    nn.init.constant_(module.weight, 1)
                    nn.init.constant_(module.bias, 0)

    def fit_kmeans(self, train_loader):
        """训练K-means聚类器"""
        
        # 提取所有训练数据
        X_all = []
        for batch in train_loader:
            X_batch = batch[0].cpu().numpy()
            X_all.append(X_batch)
        
        X_all = np.concatenate(X_all, axis=0)
        
        # 训练K-means
        self.cluster_labels = self.kmeans.fit_predict(X_all)
        self.is_fitted = True
        
        # 打印聚类信息
        unique, counts = np.unique(self.cluster_labels, return_counts=True)
        print(f"K-means clustering completed:")
        for cluster_id, count in zip(unique, counts):
            print(f"  Cluster {cluster_id}: {count} samples ({count/len(X_all)*100:.1f}%)")
        
        return self.cluster_labels

    def create_cluster_dataloaders(self, original_dataloader):
        """为每个聚类创建独立的DataLoader"""
        if not self.is_fitted:
            self.fit_kmeans(original_dataloader)
        
        from torch.utils.data import TensorDataset, DataLoader
        
        # 提取所有数据
        all_data = []
        for batch in original_dataloader:
            all_data.append(batch)
        
        # 按聚类分组数据
        cluster_dataloaders = {}
        
        for cluster_id in range(self.n_clusters):
            cluster_samples = []
            sample_idx = 0
            
            for batch in all_data:
                batch_size = batch[0].size(0)
                
                for i in range(batch_size):
                    if sample_idx + i < len(self.cluster_labels):
                        if self.cluster_labels[sample_idx + i] == cluster_id:
                            # 收集该样本
                            sample = [batch[j][i] for j in range(len(batch))]
                            cluster_samples.append(sample)
                
                sample_idx += batch_size
            
            if cluster_samples:
                # 创建该聚类的TensorDataset
                cluster_tensors = []
                for j in range(len(cluster_samples[0])):
                    tensor_data = torch.stack([sample[j] for sample in cluster_samples])
                    cluster_tensors.append(tensor_data)
                
                cluster_dataset = TensorDataset(*cluster_tensors)
                cluster_dataloader = DataLoader(
                    cluster_dataset, 
                    batch_size=min(len(cluster_samples), original_dataloader.batch_size),
                    shuffle=True,
                    drop_last=True
                )
                cluster_dataloaders[cluster_id] = cluster_dataloader
                # print(f"Cluster {cluster_id}: Created dataloader with {len(cluster_samples)} samples")
            else:
                pass
                # print(f"Cluster {cluster_id}: No samples found")
    
        return cluster_dataloaders

    def train_step(self, train_loader, criterion=None):
        """训练步骤"""
        if not self.is_fitted:
            self.fit_kmeans(train_loader)
        
        if criterion is None:
            criterion = self.loss_fn
        elif criterion == 'elastic_net':
            criterion = self.elastic_net_loss
        else:
            raise ValueError("Unsupported criterion. Supported values are 'elastic_net' or None (default MSE).")
        
        self.train()
        total_loss = 0.0
        total_samples = 0
        
        # 创建每个聚类的DataLoader（只在第一次调用时创建）
        if not hasattr(self, 'cluster_dataloaders') or not self.cluster_dataloaders:
            self.cluster_dataloaders = self.create_cluster_dataloaders(train_loader)
        

        # 为每个聚类分别训练
        active_clusters = 0
        for cluster_id in range(self.n_clusters):
            if cluster_id not in self.cluster_dataloaders:
                continue
                
            cluster_dataloader = self.cluster_dataloaders[cluster_id]
            cluster_model = self.cluster_models[f'cluster_{cluster_id}']
            
            if cluster_model is None:
                print(f"Warning: Model for cluster {cluster_id} is None")
                continue
                
            cluster_loss = 0.0
            cluster_samples = 0
            
            for batch in cluster_dataloader:
                inputs = batch[0].to(self.device)
                targets = batch[self.target].to(self.device)
                
                if targets.dim() == 1:
                    targets = targets.unsqueeze(1)
                
                self.optimizer.zero_grad()
                
                try:
                    # 使用对应聚类的模型
                    outputs = cluster_model(inputs)
                    loss = criterion(outputs, targets)
                    
                    loss.backward()
                    self.optimizer.step()
                    
                    cluster_loss += loss.item()
                    cluster_samples += len(inputs)
                    
                except Exception as e:
                    print(f"Error training cluster {cluster_id}: {e}")
                    continue
            
            if cluster_samples > 0:
                total_loss += cluster_loss
                total_samples += cluster_samples
                active_clusters += 1
        
        if active_clusters == 0:
            print("Warning: No active clusters found for training")
            return 0.0
            
        return total_loss / max(total_samples, 1)

    def forward(self, x, cluster_id=None):
        """前向传播，聚类模型输出后再经过全连接层"""
        if cluster_id is not None:
            cluster_model = self.cluster_models[f'cluster_{cluster_id}']
            if cluster_model is None:
                raise ValueError(f"Model for cluster {cluster_id} is None")
            out = cluster_model(x)
            return out
        else:
            if not self.is_fitted:
                raise ValueError("K-means not fitted. Please call fit_kmeans first.")
            x_np = x.cpu().numpy()
            predicted_clusters = self.kmeans.predict(x_np)
            outputs = []
            for i, cluster in enumerate(predicted_clusters):
                sample_input = x[i:i+1]
                cluster_model = self.cluster_models[f'cluster_{cluster}']
                if cluster_model is None:
                    raise ValueError(f"Model for cluster {cluster} is None")
                out = cluster_model(sample_input)
                outputs.append(out)
            return torch.cat(outputs, dim=0)

    def predict(self, test_loader, label_mean, label_std):
        """预测方法"""
        self.eval()
        pred_list = []
        act_list = []
        
        with torch.no_grad():
            for test_data in test_loader:
                data = test_data[0].to(self.device)
                
                # 提取实际值
                act_high = test_data[1].to(self.device)
                act_low = test_data[2].to(self.device)
                act_close = test_data[3].to(self.device)
                
                if act_high.dim() == 1:
                    act_high = act_high.unsqueeze(1)
                if act_low.dim() == 1:
                    act_low = act_low.unsqueeze(1)
                if act_close.dim() == 1:
                    act_close = act_close.unsqueeze(1)
                
                act = torch.cat([act_high, act_low, act_close], dim=1)
                act = act * label_std + label_mean
                act_list.append(act.cpu().numpy())
                
                # 预测
                pred = self.forward(data)
                pred = pred * label_std + label_mean
                pred_list.append(pred.cpu().numpy())
        
        pred = np.concatenate(pred_list, axis=0)
        act = np.concatenate(act_list, axis=0)
        return pred, act

    def fit(self, train_loader, epochs=50, resume_training=False, patience=5, criterion=None):
        """训练主函数"""
        
        # 如果需要恢复训练，先加载checkpoint
        start_epoch = 0
        if resume_training:
            start_epoch = self._load_checkpoint()
            print(f"Resuming training from epoch {start_epoch + 1}")
        else:
            # 首先训练K-means（只在新训练时）
            if not self.is_fitted:
                self.fit_kmeans(train_loader)
        
        no_improvement_count = 0
        prev_loss = float('inf')

        
        for epoch in tqdm(range(start_epoch, epochs), colour='#FA6780'):
            train_loss = self.train_step(train_loader, criterion=criterion)
            
            # 每10个epoch保存checkpoint和打印信息
            if (epoch + 1) % 10 == 0:
                self._save_checkpoint(epoch)
                print(f"Epoch [{epoch + 1}/{epochs}], Loss: {train_loss:.4f}")
            
            # 早停机制
            if round(train_loss, 4) < round(prev_loss, 4) - 0.001:
                prev_loss = train_loss
                no_improvement_count = 0
                # 保存最佳模型
                self.save_model()
            else:
                no_improvement_count += 1
                if no_improvement_count >= patience:
                    break

        # 训练完成后清理状态
        self.is_fitted = False
        if hasattr(self, 'cluster_dataloaders'):
            delattr(self, 'cluster_dataloaders')

    def reset_model(self):
        """重置模型"""
        # 重新创建聚类模型
        self.cluster_models = nn.ModuleDict()
        for i in range(self.n_clusters):
            self.cluster_models[f'cluster_{i}'] = self._create_network(self.input_dim, self.output_dim, self.layer)
        
        self._initialize_weights()
        self.optimizer = optim.Adam(self.parameters(), lr=0.01)
        self.is_fitted = False
        self.cluster_labels = None
        
        # 清除缓存的cluster dataloaders
        if hasattr(self, 'cluster_dataloaders'):
            delattr(self, 'cluster_dataloaders')
        
        # 移动到设备
        self.to(self.device)

    def save_model(self, path=None):
        """保存最终模型"""
        if path is None:
            path = self.model_path
        
        save_dict = {
            'state_dict': self.state_dict(),
            'kmeans': self.kmeans,
            'cluster_labels': self.cluster_labels,
            'is_fitted': self.is_fitted,
            'n_clusters': self.n_clusters,
            'model_name': self.model_name,
            'input_dim': self.input_dim,
            'output_dim': self.output_dim,
            'layer': self.layer
        }
        
        os.makedirs(os.path.dirname(path), exist_ok=True)
        torch.save(save_dict, path)

    def load_model(self, path=None):
        """加载最终模型"""
        if path is None:
            path = self.model_path
        
        if os.path.exists(path):
            try:
                save_dict = torch.load(path, map_location=self.device, weights_only=False)
                
                self.load_state_dict(save_dict['state_dict'])
                self.kmeans = save_dict['kmeans']
                self.cluster_labels = save_dict['cluster_labels']
                self.is_fitted = save_dict['is_fitted']
                
                return True
            except Exception as e:
                print(f"Error loading model from {path}: {e}")
                return False
        else:
            print(f"Model file {path} not found")
            return False

    def get_cluster_info(self):
        """获取聚类信息"""
        if not self.is_fitted:
            return "K-means not fitted yet"
        
        unique, counts = np.unique(self.cluster_labels, return_counts=True)
        info = f"K-means clustering with {self.n_clusters} clusters:\n"
        for cluster_id, count in zip(unique, counts):
            info += f"  Cluster {cluster_id}: {count} samples ({count/len(self.cluster_labels)*100:.1f}%)\n"
        return info
    
    def _save_checkpoint(self, epoch):
        """保存训练检查点"""
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'loss': self.loss_fn,
            'kmeans': self.kmeans,
            'cluster_labels': self.cluster_labels,
            'is_fitted': self.is_fitted,
            'n_clusters': self.n_clusters,
            'model_name': self.model_name
        }
        
        os.makedirs(os.path.dirname(self.checkpoint_path), exist_ok=True)
        torch.save(checkpoint, self.checkpoint_path)

    def _load_checkpoint(self):
        """加载训练检查点"""
        if os.path.exists(self.checkpoint_path):
            try:
                checkpoint = torch.load(self.checkpoint_path, map_location=self.device, weights_only=False)
                
                # 加载模型状态
                self.load_state_dict(checkpoint['model_state_dict'])
                self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
                
                # 加载K-means相关信息
                self.kmeans = checkpoint['kmeans']
                self.cluster_labels = checkpoint['cluster_labels']
                self.is_fitted = checkpoint['is_fitted']
                
                epoch = checkpoint['epoch']
                return epoch
                
            except Exception as e:
                print(f"Error loading checkpoint: {e}")
                print("Starting training from scratch.")
                return 0
        else:
            print("No checkpoint found. Starting from scratch.")
            return 0
    
    def evaluate(self, test_loader):
        """nondemeaned R² evaluation"""
        self.eval()
        total_sse = 0.0
        total_ss = 0.0
        with torch.no_grad():
            for test_data in test_loader:
                inputs = test_data[0].to(self.device)
                targets = test_data[self.target].to(self.device)
                if targets.dim() == 1:
                    targets = targets.unsqueeze(1)
                outputs = self(inputs)
                # targets >=0
                targets = torch.clamp(targets, min=0)
                outputs = torch.clamp(outputs, min=0)
                # Calculate sum of squared errors and total sum of squares
                sse = torch.sum((targets - outputs) ** 2)
                ss = torch.sum(targets ** 2)
                total_sse += sse.item()
                total_ss += ss.item()
        if total_ss < 1e-8:
            return 0
        else:
            return 1 - total_sse / total_ss
        
    def elastic_net_loss(self, outputs, targets):
        """Elastic net损失函数"""
        mse_loss = self.loss_fn(outputs, targets)
        l1_reg = 0
        l2_reg = 0
        for module in self.modules():
            if isinstance(module, nn.Linear):
                l1_reg += torch.sum(torch.abs(module.weight))
                l2_reg += torch.sum(module.weight ** 2)
        
        elastic_reg = self.alpha * (self.l1_ratio * l1_reg + (1 - self.l1_ratio) * l2_reg)
        return mse_loss + elastic_reg


class CNN(nn.Module):
    def __init__(self, input_dim, output_dim=1, target=3, alpha=0.8, l1_ratio=0.5, model_name="CNN"):
        super(CNN, self).__init__()
        
        import math
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.target = target
        self.model_name = model_name
        
        # 计算最接近正方形的尺寸
        self.height = int(math.sqrt(input_dim))
        self.width = int(math.ceil(input_dim / self.height))
        self.padded_size = self.height * self.width
        
        # 卷积层
        self.conv_layers = nn.Sequential(
            # 第一个卷积块
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # 第二个卷积块
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # 第三个卷积块
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4))  # 自适应池化到固定尺寸
        )
        
        # 全连接层
        self.fc_layers = nn.Sequential(
            nn.Linear(64 * 4 * 4, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, output_dim)
        )
        
        self._initialize_weights()
        self.optimizer = optim.Adam(self.parameters(), lr=0.001)
        self.loss_fn = nn.MSELoss()
        self.alpha = alpha
        self.l1_ratio = l1_ratio
        
        # 设备设置
        if torch.cuda.is_available():
            self.device = torch.device("cuda")
        else:
            self.device = torch.device("cpu")
        self.to(self.device)
        
        # 模型保存路径
        if not os.path.exists("model/checkpoints"):
            os.makedirs("model/checkpoints")
        if not os.path.exists("model/final_models"):
            os.makedirs("model/final_models")
        self.checkpoint_path = f"model/checkpoints/{self.model_name}_checkpoint.pth"
        self.model_path = f"model/final_models/{self.model_name}.pth"

    def _initialize_weights(self):
        """初始化权重"""
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)
            elif isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0, std=0.1)
                nn.init.constant_(module.bias, 0)
            elif isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d)):
                nn.init.constant_(module.weight, 1)
                nn.init.constant_(module.bias, 0)

    def forward(self, x):
        """前向传播"""
        batch_size = x.size(0)
        

        if x.size(1) < self.padded_size:
            padding = torch.zeros(batch_size, self.padded_size - x.size(1), device=x.device)
            x = torch.cat([x, padding], dim=1)
        elif x.size(1) > self.padded_size:
            x = x[:, :self.padded_size]

        x = x.view(batch_size, 1, self.height, self.width)
        x = self.conv_layers(x)
        x = x.view(batch_size, -1)
        x = self.fc_layers(x)
        
        return x

    def elastic_net_loss(self, outputs, targets):
            """弹性网络损失函数：MSE + L1正则化 + L2正则化"""
            mse_loss = self.loss_fn(outputs, targets)
            l1_reg = 0
            l2_reg = 0
            
            # 对卷积层和全连接层都应用正则化
            for module in self.modules():
                if isinstance(module, (nn.Conv2d, nn.Linear)):
                    l1_reg += torch.sum(torch.abs(module.weight))
                    l2_reg += torch.sum(module.weight ** 2)
            
            # 弹性网络正则化项
            elastic_reg = self.alpha * (self.l1_ratio * l1_reg + (1 - self.l1_ratio) * l2_reg)
            
            return mse_loss + elastic_reg
    
    def train_step(self, train_loader, criterion=None):
        """训练步骤"""
        if criterion is None:
            criterion = self.loss_fn
        elif criterion == 'elastic_net':
            criterion = self.elastic_net_loss
        else:
            pass
            
        
        self.train()
        total_loss = 0.0
        
        for train_data in train_loader:
            inputs = train_data[0].to(self.device)
            targets = train_data[self.target].to(self.device)
            
            if targets.dim() == 1:
                targets = targets.unsqueeze(1)
            
            self.optimizer.zero_grad()
            outputs = self.forward(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item()
        
        return total_loss / len(train_loader)

    def fit(self, train_loader, epochs=100, resume_training=False, patience=5, criterion=None):
        """训练主函数"""
        start_epoch = 0
        no_improvement_count = 0
        
        if resume_training:
            start_epoch = self._load_checkpoint()
            print(f"Resuming training from epoch {start_epoch + 1}")

        prev_loss = float('inf')
        for epoch in tqdm(range(start_epoch, epochs), colour='#FA6780'):
            train_loss = self.train_step(train_loader, criterion)
            
            # 每10个epoch保存checkpoint
            if (epoch + 1) % 10 == 0:
                self._save_checkpoint(epoch)
                print(f"Epoch [{epoch + 1}/{epochs}], Loss: {train_loss:.4f}")

            # 早停机制
            if round(train_loss, 4) < round(prev_loss, 4) - 0.001:
                prev_loss = train_loss
                no_improvement_count = 0
                self.save_model()
            else:
                no_improvement_count += 1
                if no_improvement_count >= patience:
                    break

    def predict(self, test_loader, label_mean, label_std):
        """预测方法"""
        self.eval()
        pred_list = []
        act_list = []
        
        with torch.no_grad():
            for test_data in test_loader:
                data = test_data[0].to(self.device)
                act_high = test_data[1].to(self.device)
                act_low = test_data[2].to(self.device)
                act_close = test_data[3].to(self.device)
                
                if act_high.dim() == 1:
                    act_high = act_high.unsqueeze(1)
                if act_low.dim() == 1:
                    act_low = act_low.unsqueeze(1)
                if act_close.dim() == 1:
                    act_close = act_close.unsqueeze(1)
                
                act = torch.cat([act_high, act_low, act_close], dim=1)
                act = act * label_std + label_mean
                act_list.append(act.cpu().numpy())
                
                pred = self(data)
                pred = pred * label_std + label_mean
                pred_list.append(pred.cpu().numpy())
        
        pred = np.concatenate(pred_list, axis=0)
        act = np.concatenate(act_list, axis=0)
        return pred, act

    def evaluate(self, test_loader):
        """nondemeaned R² evaluation"""
        self.eval()
        total_sse = 0.0
        total_ss = 0.0
        
        with torch.no_grad():
            for test_data in test_loader:
                inputs = test_data[0].to(self.device)
                targets = test_data[self.target].to(self.device)
                
                if targets.dim() == 1:
                    targets = targets.unsqueeze(1)
                
                outputs = self(inputs)
                
                # 确保非负
                targets = torch.clamp(targets, min=0)
                outputs = torch.clamp(outputs, min=0)
                
                # 计算误差
                sse = torch.sum((targets - outputs) ** 2)
                ss = torch.sum(targets ** 2)
                total_sse += sse.item()
                total_ss += ss.item()
        
        if total_ss < 1e-8:
            return 0
        else:
            return 1 - total_sse / total_ss

    def reset_model(self):
        """重置模型"""
        self._initialize_weights()
        self.optimizer = optim.Adam(self.parameters(), lr=0.001)

    def save_model(self, path=None):
        """保存模型"""
        if path is None:
            path = self.model_path
        os.makedirs(os.path.dirname(path), exist_ok=True)
        torch.save(self.state_dict(), path)

    def load_model(self, path=None):
        """加载模型"""
        if path is None:
            path = self.model_path
        self.load_state_dict(torch.load(path, map_location=self.device))

    def _save_checkpoint(self, epoch):
        """保存检查点"""
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'loss': self.loss_fn
        }
        torch.save(checkpoint, self.checkpoint_path)

    def _load_checkpoint(self):
        """加载检查点"""
        if os.path.exists(self.checkpoint_path):
            checkpoint = torch.load(self.checkpoint_path, weights_only=False, map_location=self.device)
            self.load_state_dict(checkpoint['model_state_dict'])
            self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            return checkpoint['epoch']
        else:
            print("No checkpoint found. Starting from scratch.")
            return 0

# 创建支持向量机模型
# 创建支持向量机模型
class SVM:
    def __init__(self, target=3, kernel='rbf', C=1.0, gamma='scale', epsilon=0.1, 
                 model_name="SVM"):
        """
        支持向量机回归模型
        
        参数:
            target: 目标列索引 (默认3)
            kernel: 核函数类型 ('linear', 'poly', 'rbf', 'sigmoid') (默认'rbf')
            C: 正则化参数 (默认1.0)
            gamma: 核函数系数 ('scale', 'auto', 或浮点数) (默认'scale')
            epsilon: epsilon-SVR的epsilon参数 (默认0.1)
            model_name: 模型名称 (默认"SVM")
        """
        from sklearn.svm import SVR
        
        self.model = SVR(
            kernel=kernel,
            C=C,
            gamma=gamma,
            epsilon=epsilon,
            cache_size=200,  # 缓存大小(MB)
            max_iter=-1      # 最大迭代次数(-1表示无限制)
        )
        
        self.model_name = model_name
        self.target = target
        self.feature_names = None
        
        # 创建模型保存目录
        os.makedirs("model/checkpoints", exist_ok=True)
        os.makedirs("model/final_models", exist_ok=True)
            
        self.checkpoint_path = f"model/checkpoints/{self.model_name}_checkpoint.pkl"
        self.model_path = f"model/final_models/{self.model_name}.pkl"

    def fit(self, train_loader, epochs=None, patience=None, criterion=None):
        """
        训练SVM模型
        
        参数:
            train_loader: 数据加载器 (PyTorch DataLoader格式)
            epochs: 为了接口兼容保留 (SVM不需要)
            patience: 为了接口兼容保留
            criterion: 为了接口兼容保留
        """
        X_train, y_train = self._extract_data_from_loader(train_loader)
        
        # 数据标准化（SVM对特征范围敏感）
        from sklearn.preprocessing import StandardScaler
        self.scaler = StandardScaler()
        X_train_scaled = self.scaler.fit_transform(X_train)
        
        # 训练模型
        self.model.fit(X_train_scaled, y_train)
        self.save_model()
        print(f"SVM trained with {len(X_train)} samples")

    def predict(self, test_loader, label_mean=None, label_std=None):
        """
        预测并返回与PyTorch模型相同的格式
        
        参数:
            test_loader: 测试数据加载器
            label_mean: 标准化均值 (可选)
            label_std: 标准化标准差 (可选)
            
        返回:
            (pred, y_test) 元组，与PyTorch模型输出格式一致
        """
        X_test, y_test = self._extract_data_from_loader(test_loader)
        
        # 使用训练时的scaler进行标准化
        if hasattr(self, 'scaler'):
            X_test_scaled = self.scaler.transform(X_test)
        else:
            print("Warning: No scaler found. Using raw features.")
            X_test_scaled = X_test
        
        pred = self.model.predict(X_test_scaled)

        # 转换为与PyTorch模型相同的格式
        if pred.ndim == 1:
            pred = pred.reshape(-1, 1)
        if y_test.ndim == 1:
            y_test = y_test.reshape(-1, 1)
        
        # 反标准化处理
        if label_mean is not None and label_std is not None:
            if hasattr(label_mean, 'cpu'):  # 如果是PyTorch张量
                label_mean = label_mean.cpu().numpy()
            if hasattr(label_std, 'cpu'):
                label_std = label_std.cpu().numpy()
            
            # 确保维度匹配
            if label_mean.ndim == 0:
                label_mean = label_mean.item()
            if label_std.ndim == 0:
                label_std = label_std.item()
            
            pred = pred * label_std + label_mean
            y_test = y_test * label_std + label_mean
            
        return pred, y_test

    def evaluate(self, test_loader):
        """
        计算nondemeaned R²分数
        
        参数:
            test_loader: 测试数据加载器
            
        返回:
            R²分数
        """
        X_test, y_test = self._extract_data_from_loader(test_loader)
        
        # 使用训练时的scaler进行标准化
        if hasattr(self, 'scaler'):
            X_test_scaled = self.scaler.transform(X_test)
        else:
            X_test_scaled = X_test
        
        pred = self.model.predict(X_test_scaled)
        
        # 确保非负
        y_test = np.clip(y_test, a_min=0, a_max=None)
        pred = np.clip(pred, a_min=0, a_max=None)
        
        # 计算nondemeaned R²
        sse = np.sum((y_test - pred) ** 2)
        ss = np.sum(y_test ** 2)
        
        return 0 if ss < 1e-8 else 1 - sse / ss

    def _extract_data_from_loader(self, data_loader):
        """
        从PyTorch DataLoader中提取数据
        
        参数:
            data_loader: PyTorch DataLoader
            
        返回:
            (X, y) 元组
        """
        X_list, y_list = [], []
        
        for batch in data_loader:
            X_list.append(batch[0].numpy())
            y_list.append(batch[self.target].numpy())
        
        X = np.concatenate(X_list, axis=0)
        y = np.concatenate(y_list, axis=0).ravel()  # SVM需要一维目标
        
        return X, y

    def reset_model(self):
        """重置模型（重新初始化）"""
        from sklearn.svm import SVR
        
        # 保存原始参数
        params = self.model.get_params()
        self.model = SVR(**params)
        
        # 重置scaler
        if hasattr(self, 'scaler'):
            delattr(self, 'scaler')

    def save_model(self, path=None):
        """
        保存模型到文件
        
        参数:
            path: 自定义保存路径 (可选)
        """
        path = path or self.model_path
        os.makedirs(os.path.dirname(path), exist_ok=True)
        
        # 保存模型和scaler
        save_dict = {
            'model': self.model,
            'scaler': getattr(self, 'scaler', None),
            'model_name': self.model_name,
            'target': self.target
        }
        
        joblib.dump(save_dict, path)
        # print(f"SVM model successfully saved to {path}")

    def load_model(self, path=None):
        """
        从文件加载模型
        
        参数:
            path: 自定义加载路径 (可选)
        """
        path = path or self.model_path
        
        if os.path.exists(path):
            save_dict = joblib.load(path)
            self.model = save_dict['model']
            
            if save_dict['scaler'] is not None:
                self.scaler = save_dict['scaler']
            
            # print(f"SVM model successfully loaded from {path}")
        else:
            print(f"Model file {path} not found")


class Transformer(nn.Module):
    def __init__(self, input_dim, output_dim=1, target=3, seq_len=20, d_model=64, 
                 nhead=8, num_layers=3, dim_feedforward=256, dropout=0.1, 
                 alpha=0.8, l1_ratio=0.5, model_name="Transformer"):
        """
        Transformer模型用于金融时序预测
        
        参数:
            input_dim: 输入特征维度
            output_dim: 输出维度 (默认1)
            target: 目标列索引 (默认3)
            seq_len: 序列长度 (默认20)
            d_model: 模型维度 (默认64)
            nhead: 注意力头数 (默认8)
            num_layers: Transformer层数 (默认3)
            dim_feedforward: 前馈网络维度 (默认256)
            dropout: Dropout比例 (默认0.1)
            alpha: 弹性网络正则化强度 (默认0.8)
            l1_ratio: L1正则化比例 (默认0.5)
            model_name: 模型名称 (默认"Transformer")
        """
        super(Transformer, self).__init__()
        
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.target = target
        self.seq_len = seq_len
        self.d_model = d_model
        self.model_name = f"{model_name}_{num_layers}Layers"
        self.alpha = alpha
        self.l1_ratio = l1_ratio
        
        # 输入映射层
        self.input_projection = nn.Linear(input_dim, d_model)
        
        # 位置编码
        self.positional_encoding = PositionalEncoding(d_model, dropout, max_len=seq_len)
        
        # Transformer编码器
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='relu',
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, 
            num_layers=num_layers
        )
        
        # 输出层
        self.output_projection = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, dim_feedforward // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward // 2, output_dim)
        )
        
        # 初始化
        self._initialize_weights()
        self.optimizer = optim.Adam(self.parameters(), lr=0.001)
        self.loss_fn = nn.MSELoss()
        
        # 设备设置
        if torch.cuda.is_available():
            self.device = torch.device("cuda")
        else:
            self.device = torch.device("cpu")
        self.to(self.device)
        
        # 模型保存路径
        if not os.path.exists("model/checkpoints"):
            os.makedirs("model/checkpoints")
        if not os.path.exists("model/final_models"):
            os.makedirs("model/final_models")
        self.checkpoint_path = f"model/checkpoints/{self.model_name}_checkpoint.pth"
        self.model_path = f"model/final_models/{self.model_name}.pth"

    def _initialize_weights(self):
        """初始化权重"""
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)
            elif isinstance(module, nn.LayerNorm):
                nn.init.constant_(module.weight, 1)
                nn.init.constant_(module.bias, 0)

    def forward(self, x):
        """
        前向传播
        
        参数:
            x: 输入张量 (batch_size, seq_len, input_dim) 或 (batch_size, input_dim)
        """
        batch_size = x.size(0)
        
        # 如果输入是2D，重塑为序列格式
        if x.dim() == 2:
            # 将单个样本扩展为序列 (重复最后一个时间步)
            x = x.unsqueeze(1).repeat(1, self.seq_len, 1)
        
        # 确保序列长度正确
        if x.size(1) != self.seq_len:
            if x.size(1) < self.seq_len:
                # 如果序列太短，用最后一个时间步填充
                padding = x[:, -1:, :].repeat(1, self.seq_len - x.size(1), 1)
                x = torch.cat([x, padding], dim=1)
            else:
                # 如果序列太长，截取最后seq_len个时间步
                x = x[:, -self.seq_len:, :]
        
        # 输入投影: (batch_size, seq_len, input_dim) -> (batch_size, seq_len, d_model)
        x = self.input_projection(x)
        
        # 添加位置编码
        x = self.positional_encoding(x)
        
        # Transformer编码器
        x = self.transformer_encoder(x)
        
        # 使用最后一个时间步的输出
        x = x[:, -1, :]  # (batch_size, d_model)
        
        # 输出投影
        x = self.output_projection(x)
        
        return x

    def elastic_net_loss(self, outputs, targets):
        """弹性网络损失函数：MSE + L1正则化 + L2正则化"""
        mse_loss = self.loss_fn(outputs, targets)
        l1_reg = 0
        l2_reg = 0
        
        # 对所有线性层应用正则化
        for module in self.modules():
            if isinstance(module, nn.Linear):
                l1_reg += torch.sum(torch.abs(module.weight))
                l2_reg += torch.sum(module.weight ** 2)
        
        # 弹性网络正则化项
        elastic_reg = self.alpha * (self.l1_ratio * l1_reg + (1 - self.l1_ratio) * l2_reg)
        
        return mse_loss + elastic_reg

    def train_step(self, train_loader, criterion=None):
        """训练步骤"""
        if criterion is None:
            criterion = self.loss_fn
        elif criterion == 'elastic_net':
            criterion = self.elastic_net_loss
        else:
            pass
        
        self.train()
        total_loss = 0.0
        
        for train_data in train_loader:
            inputs = train_data[0].to(self.device)
            targets = train_data[self.target].to(self.device)
            
            if targets.dim() == 1:
                targets = targets.unsqueeze(1)
            
            self.optimizer.zero_grad()
            outputs = self.forward(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            
            # 梯度裁剪，防止梯度爆炸
            torch.nn.utils.clip_grad_norm_(self.parameters(), max_norm=1.0)
            
            self.optimizer.step()
            total_loss += loss.item()
        
        return total_loss / len(train_loader)

    def fit(self, train_loader, epochs=100, resume_training=False, patience=5, criterion=None):
        """训练主函数"""
        start_epoch = 0
        no_improvement_count = 0
        
        if resume_training:
            start_epoch = self._load_checkpoint()
            print(f"Resuming training from epoch {start_epoch + 1}")

        prev_loss = float('inf')
        for epoch in tqdm(range(start_epoch, epochs), colour='#FA6780'):
            train_loss = self.train_step(train_loader, criterion)
            
            # 每10个epoch保存checkpoint
            if (epoch + 1) % 10 == 0:
                self._save_checkpoint(epoch)
                print(f"Epoch [{epoch + 1}/{epochs}], Loss: {train_loss:.4f}")

            # 早停机制
            if round(train_loss, 4) < round(prev_loss, 4) - 0.001:
                prev_loss = train_loss
                no_improvement_count = 0
                self.save_model()
            else:
                no_improvement_count += 1
                if no_improvement_count >= patience:
                    break

    def predict(self, test_loader, label_mean, label_std):
        """预测方法"""
        self.eval()
        pred_list = []
        act_list = []
        
        with torch.no_grad():
            for test_data in test_loader:
                data = test_data[0].to(self.device)
                act_high = test_data[1].to(self.device)
                act_low = test_data[2].to(self.device)
                act_close = test_data[3].to(self.device)
                
                if act_high.dim() == 1:
                    act_high = act_high.unsqueeze(1)
                if act_low.dim() == 1:
                    act_low = act_low.unsqueeze(1)
                if act_close.dim() == 1:
                    act_close = act_close.unsqueeze(1)
                
                act = torch.cat([act_high, act_low, act_close], dim=1)
                act = act * label_std + label_mean
                act_list.append(act.cpu().numpy())
                
                pred = self(data)
                pred = pred * label_std + label_mean
                pred_list.append(pred.cpu().numpy())
        
        pred = np.concatenate(pred_list, axis=0)
        act = np.concatenate(act_list, axis=0)
        return pred, act

    def evaluate(self, test_loader):
        """nondemeaned R² evaluation"""
        self.eval()
        total_sse = 0.0
        total_ss = 0.0
        
        with torch.no_grad():
            for test_data in test_loader:
                inputs = test_data[0].to(self.device)
                targets = test_data[self.target].to(self.device)
                
                if targets.dim() == 1:
                    targets = targets.unsqueeze(1)
                
                outputs = self(inputs)
                
                # 确保非负
                targets = torch.clamp(targets, min=0)
                outputs = torch.clamp(outputs, min=0)
                
                # 计算误差
                sse = torch.sum((targets - outputs) ** 2)
                ss = torch.sum(targets ** 2)
                total_sse += sse.item()
                total_ss += ss.item()
        
        if total_ss < 1e-8:
            return 0
        else:
            return 1 - total_sse / total_ss

    def reset_model(self):
        """重置模型"""
        self._initialize_weights()
        self.optimizer = optim.Adam(self.parameters(), lr=0.001)

    def save_model(self, path=None):
        """保存模型"""
        if path is None:
            path = self.model_path
        os.makedirs(os.path.dirname(path), exist_ok=True)
        torch.save(self.state_dict(), path)

    def load_model(self, path=None):
        """加载模型"""
        if path is None:
            path = self.model_path
        self.load_state_dict(torch.load(path, map_location=self.device))

    def _save_checkpoint(self, epoch):
        """保存检查点"""
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'loss': self.loss_fn
        }
        torch.save(checkpoint, self.checkpoint_path)

    def _load_checkpoint(self):
        """加载检查点"""
        if os.path.exists(self.checkpoint_path):
            checkpoint = torch.load(self.checkpoint_path, weights_only=False, map_location=self.device)
            self.load_state_dict(checkpoint['model_state_dict'])
            self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            return checkpoint['epoch']
        else:
            print("No checkpoint found. Starting from scratch.")
            return 0



In [ ]:
# rollingtraintest///////////////////////////////////////////////////////////////////
class RollingTrainTest:
    def __init__(self, model, Data, train_size=0.5, test_size=0.1, epochs=50, patience=5, criterion=None, count = 0):
        self.model = model
        self.model_name = self.model.model_name
        self.Data = Data
        self.train_size_original = train_size
        self.train_size = train_size
        self.test_size = test_size
        self.epochs = epochs
        self.patience = patience
        self.criterion = criterion
        self.count = count
        if not os.path.exists("CSV"):
            os.makedirs("CSV")

    def info(self, predictability_name):
        self.predictability_name = predictability_name
        return

    def run(self):
        self.predictability = []
        self.pred_list = []
        self.pred_list_top10 = []
        self.act_list = []
        self.num_iterations = math.ceil((1 - self.train_size) / self.test_size)
        print (f'{self.model_name} will run {self.num_iterations} iterations.')
        all_start_time = time.time()
        for _ in range(self.num_iterations):
            start_time = time.time()
            self.train_loader = self.Data.get_train_loader(train_size=self.train_size)
            self.test_loader = self.Data.get_test_loader(test_size=self.test_size)
            self.model.fit(self.train_loader, epochs=self.epochs, patience=self.patience, criterion=self.criterion)
            self.model.load_model()  # Load the best model after training
            result= round(self.model.evaluate(self.test_loader),4)
            result_train = round(self.model.evaluate(self.train_loader),4)
            label_mean = self.Data.dataloader.label_mean
            label_std = self.Data.dataloader.label_std
            pred, act = self.model.predict(self.test_loader, label_mean, label_std)
            # store the predictions
            self.pred_list.append(pred)
            # keep the top 10 and bottom 10 predictions, change the others to 0, keep the order
            long_indices = np.argsort(pred)[-10:]
            short_indices = np.argsort(pred)[:10]
            pred_top10 = np.zeros_like(pred)
            pred_top10[long_indices] = pred[long_indices]
            pred_top10[short_indices] = pred[short_indices]
            self.pred_list_top10.append(pred_top10)
            # store the actual values
            self.act_list.append(act)
            self.predictability.append(result)
            # return the spent seconds
            spent_time = round(time.time() - start_time, 2)
            print (f'No.{_+1} Test Predictability: {result:.4f}')
            print (f'No.{_+1} Train Predictability: {result_train:.4f}')
            # save test predictability and train predictability
            file = f'CSV/predictability_{self.model_name}.csv'
            mode = 'a' if os.path.exists(file) else 'w'
            with open(file, mode) as f:
                if mode == 'w':
                    f.write(f'model_name,No.,test_predictability,train_predictability,spent_time\n')
                if _ == 0:
                    f.write(f'{self.predictability_name}\n')
                f.write(f'{self.model_name},{_+1},{result:.4f},{result_train:.4f},{spent_time:.2f}s\n')
            self.train_size += self.test_size
            # self.model.reset_model()
        self.pred = np.concatenate(self.pred_list, axis=0)
        self.act = np.concatenate(self.act_list, axis=0)
        self.pred_top10 = np.concatenate(self.pred_list_top10, axis=0)
        all_spent_time = round(time.time() - all_start_time, 2)
        print(f"Predictability of {self.model_name}: {sum(self.predictability) / len(self.predictability):.4f}")
        file = 'CSV/predictability.csv'
        mode = 'a' if os.path.exists(file) else 'w'
        with open(file, mode) as f:
            if mode == 'w':
                f.write('model_name,predictability,all_spent_time\n')
            if self.count == 0:
                f.write(f'{self.predictability_name}\n')
            f.write(f'{self.model_name},{sum(self.predictability) / len(self.predictability):.4f},{all_spent_time:.2f}s\n')


    def backtest(self, trade_mode=1, data_frequency='monthly'):
        if trade_mode == 1:
            pred = self.pred[:,-1]
            act = self.act[:,-1]
            pred = np.where(pred > 0, 1, 0)
            
            all_returns = pred * act
            num = math.ceil(len(all_returns) /12 /self.num_iterations)

            # 按顺序提取num个return数据,作为一个月的收益率
            # 取平均值时忽略0
            period_returns = []
            for i in range(0, len(all_returns), num):
                period_data = all_returns[i:i+num]
                non_zero_data = period_data[period_data != 0]
                if len(non_zero_data) > 0:
                    period_returns.append(np.mean(non_zero_data))
                else:
                    period_returns.append(0)
            # 保留4位小数
            period_returns = [round(x, 4) for x in period_returns]
            period_returns = np.array(period_returns)
            # 计算累计收益率
            cum_returns = np.cumprod(1 + period_returns/100) - 1
            cum_returns = [round(x*100, 4) for x in cum_returns]

            # 根据period_returns的数量分配月份,最后一个月是2024年12月
            months = pd.date_range(end='2024-12-31', periods=len(period_returns), freq='M')
            
            # save to dataframe
            df = pd.DataFrame(
                {'Month': months,
                'returns': period_returns,
                'cum_returns': cum_returns}
            )

            file = f'CSV/predictions_{self.model_name}.csv'
            mode = 'a' if os.path.exists(file) else 'w'
            with open(file, mode, encoding='utf-8', newline="") as f:
                if mode == 'w':
                    # 如果是新文件，先写入描述信息
                    f.write('Month,returns,cum_returns\n')
                    f.write(f'{self.predictability_name}\n')
                    # 写入数据，不包含表头，去掉最后的换行符
                    csv_content = df.to_csv(index=False, header=False)
                    f.write(csv_content)
                else:
                    f.write(f'{self.predictability_name}\n')
                    csv_content = df.to_csv(index=False, header=False)
                    f.write(csv_content)

            # 使用月度数据计算风险指标
            risk_metrics = RiskMetrics(risk_free_rate=0.01, data_frequency=data_frequency)
            metrics = risk_metrics.calculate_metrics(period_returns)
            
            # 保存到CSV
            file_path = 'CSV/profit_indicators.csv'
            mode = 'a' if os.path.exists(file_path) else 'w'      
            with open(file_path, mode) as f:
                if mode == 'w':
                    f.write('model_name,mean_return,sharpe_ratio,annualized_return,annualized_volatility,max_drawdown,win_rate\n')
                if self.count == 0:
                    f.write(f'{self.predictability_name}\n')
                f.write(f'{self.model_name},{metrics["mean_return"]:.4f},{metrics["sharpe_ratio"]:.4f},{metrics["annualized_return"]:.4f},{metrics["annualized_volatility"]:.4f},{metrics["max_drawdown"]:.4f},{metrics["win_rate"]:.4f}\n')
        else:
            pass
        return

In [ ]:
factor = pd.read_csv('/kaggle/input/big-factor-label/factor_interaction_0.1_R.csv')
input_dim = factor.shape[1] - 2
print(f"Input dimension: {input_dim}")

In [ ]:
label = pd.read_csv('/kaggle/input/big-factor-label/label_cleaned.csv')
label.describe()

In [ ]:
Data = LoadData(factor, label, batch_size=32, num_workers=0, shuffle=True)
model_list = [
    NN(input_dim, target=3, alpha=0.8, l1_ratio=0.5, layer=2, model_name="NN"),
    NN(input_dim, target=3, alpha=0.8, l1_ratio=0.5, layer=3, model_name="NN"),
    NN(input_dim, target=3, alpha=0.8, l1_ratio=0.5, layer=4, model_name="NN"),
    NN(input_dim, target=3, alpha=0.8, l1_ratio=0.5, layer=5, model_name="NN"),
    K_Means_NN(input_dim, target=3, alpha=0.8, l1_ratio=0.5, n_clusters=10, layer=5, model_name="K_Means_NN"),
    CNN(input_dim, target=3, model_name="CNN")
]

In [ ]:
'''
Data = LoadData(factor, label, batch_size=32, num_workers=0, shuffle=True)
model_list = [
    Transformer(input_dim, target=3, model_name="Transformer")
]
'''

In [ ]:
count = 0
for model in model_list:
    RTT = RollingTrainTest(model, Data, train_size=0.5, test_size=0.1, epochs=10, patience=3, criterion="elastic_net", count=count)
    RTT.info(
        predictability_name = '[interaction_0.1_R | elastic_net | batch=32 | shuffle=True | alpha=0.8 | l1=0.5 | LeakyReLU(0.3) |FL_None]'
        )
    RTT.run()
    RTT.backtest(trade_mode=1)
    count += 1
    print(f"Model {model.model_name} backtest completed...")
    print("=" * 50)